# Task 2: Named Entity Recognition + Image Classification

This notebook covers exploratory data analysis, model training, evaluation, and pipeline demonstration for animal classification and NER tasks.

## 1. Load and Explore Animal Classification Dataset

In this section, we load the animal classification dataset, inspect its structure, and display basic statistics such as number of classes and samples per class.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from glob import glob
from PIL import Image

# Example: Load dataset (replace with your dataset path)
data_dir = 'animal_dataset/'  # Should contain subfolders for each animal class
classes = [d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))]
print(f"Number of classes: {len(classes)}")
print(f"Classes: {classes}")

# Count samples per class
data_stats = {cls: len(glob(os.path.join(data_dir, cls, '*'))) for cls in classes}
print("Samples per class:", data_stats)

## 2. Visualize Dataset Samples and Class Distribution

Plot sample images from each animal class and visualize the class distribution using bar charts.

In [ ]:
# Plot class distribution
plt.figure(figsize=(10,5))
plt.bar(data_stats.keys(), data_stats.values())
plt.title('Class Distribution')
plt.xlabel('Animal Class')
plt.ylabel('Number of Samples')
plt.xticks(rotation=45)
plt.show()

# Show sample images from each class
fig, axes = plt.subplots(2, 5, figsize=(15,6))
for i, cls in enumerate(classes[:10]):
    img_files = glob(os.path.join(data_dir, cls, '*'))
    img = Image.open(img_files[0])
    axes[i//5, i%5].imshow(img)
    axes[i//5, i%5].set_title(cls)
    axes[i//5, i%5].axis('off')
plt.tight_layout()
plt.show()

## 3. Prepare Data for Image Classification Model

Preprocess images (resize, normalize, split into train/validation/test sets) and encode class labels for model training.

In [ ]:
from sklearn.model_selection import train_test_split

IMG_SIZE = (128, 128)
X, y = [], []
for cls in classes:
    img_files = glob(os.path.join(data_dir, cls, '*'))
    for img_path in img_files:
        img = Image.open(img_path).resize(IMG_SIZE)
        X.append(np.array(img)/255.0)
        y.append(cls)

X = np.array(X)
y = np.array(y)

# Encode labels
y_encoded = pd.factorize(y)[0]

# Split data
X_train, X_temp, y_train, y_temp = train_test_split(X, y_encoded, test_size=0.3, random_state=42, stratify=y_encoded)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)
print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

## 4. Train Image Classification Model

Define and train a neural network or CNN for animal classification using the prepared dataset.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.InputLayer(input_shape=IMG_SIZE + (3,)),
    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(len(classes), activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

history = model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=5, batch_size=32)

## 5. Evaluate Image Classification Model

Assess model performance using metrics such as accuracy, confusion matrix, and visualize predictions.

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix

# Evaluate on test set
preds = np.argmax(model.predict(X_test), axis=1)
acc = accuracy_score(y_test, preds)
print(f"Test Accuracy: {acc:.4f}")

# Confusion matrix
cm = confusion_matrix(y_test, preds)
plt.figure(figsize=(8,6))
plt.imshow(cm, cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.colorbar()
plt.show()

# Show some predictions
for i in range(5):
    plt.imshow(X_test[i])
    plt.title(f"True: {classes[y_test[i]]}, Pred: {classes[preds[i]]}")
    plt.axis('off')
    plt.show()

## 6. Prepare Data for NER Model

Collect or simulate text samples containing animal names, annotate them for NER, and preprocess for model training.

In [ ]:
# Simulate text samples and NER annotations
text_samples = [
    "There is a cow in the picture.",
    "A dog is running in the field.",
    "I see a cat near the tree.",
    "The horse is jumping.",
    "A sheep is grazing.",
    "A lion is resting.",
    "A tiger is hunting.",
    "A bear is fishing.",
    "A fox is hiding.",
    "A wolf is howling."
]

# Annotate animal entities (BIO format)
annotations = [
    [(0, 17, 'ANIMAL')],  # cow
    [(2, 5, 'ANIMAL')],   # dog
    [(9, 12, 'ANIMAL')],  # cat
    [(4, 9, 'ANIMAL')],   # horse
    [(2, 7, 'ANIMAL')],   # sheep
    [(2, 6, 'ANIMAL')],   # lion
    [(2, 7, 'ANIMAL')],   # tiger
    [(2, 6, 'ANIMAL')],   # bear
    [(2, 5, 'ANIMAL')],   # fox
    [(2, 6, 'ANIMAL')]    # wolf
]

# Prepare for transformer NER (convert to token-level labels)
# (This is a simplified example; real NER datasets require tokenization and label alignment)

## 7. Train NER Model for Animal Title Extraction

Fine-tune a transformer-based NER model (e.g., BERT or similar) to extract animal entities from text.

In [ ]:
from transformers import BertTokenizerFast, BertForTokenClassification, Trainer, TrainingArguments
import torch

# Example: Prepare tokenized inputs (simplified)
tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')
inputs = tokenizer(text_samples, padding=True, truncation=True, return_tensors='pt')

# Dummy labels for demonstration (real labels require alignment)
labels = torch.zeros_like(inputs['input_ids'])

model = BertForTokenClassification.from_pretrained('bert-base-uncased', num_labels=2)

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=1,
    per_device_train_batch_size=2,
    logging_steps=10,
    disable_tqdm=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=None,  # Replace with real dataset
)

# trainer.train()  # Uncomment when real dataset is available

## 8. Evaluate NER Model

Evaluate the NER model using metrics like precision, recall, F1-score, and visualize entity extraction results.

In [ ]:
# Example: Evaluate NER model (dummy output)
from sklearn.metrics import precision_score, recall_score, f1_score

# Dummy predictions and labels
pred_labels = [1, 0, 1, 1, 0, 1, 1, 0, 1, 1]  # 1=animal entity, 0=other
true_labels = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

precision = precision_score(true_labels, pred_labels)
recall = recall_score(true_labels, pred_labels)
f1 = f1_score(true_labels, pred_labels)
print(f"Precision: {precision:.2f}, Recall: {recall:.2f}, F1: {f1:.2f}")

# Visualize entity extraction
for text, pred in zip(text_samples, pred_labels):
    print(f"Text: {text} | Animal detected: {bool(pred)}")

## 9. Demo: Pipeline Inference with Text and Image Inputs

Demonstrate the full pipeline: input a text and an image, extract animal title from text, classify the animal in the image, and output a boolean indicating if the text matches the image.

In [ ]:
# Demo pipeline

def pipeline(text, image):
    # NER: extract animal title
    animal_detected = None
    for animal in classes:
        if animal in text.lower():
            animal_detected = animal
            break
    # Image classification
    img_arr = np.array(image.resize(IMG_SIZE))/255.0
    img_arr = img_arr.reshape(1, *IMG_SIZE, 3)
    pred_class = classes[np.argmax(model.predict(img_arr))]
    # Output boolean
    result = (animal_detected == pred_class)
    print(f"Text animal: {animal_detected}, Image animal: {pred_class}, Match: {result}")
    return result

# Example usage:
# pipeline("There is a cat in the picture.", Image.open('animal_dataset/cat/cat1.jpg'))